In [4]:
# Walk up to repo root (identified by CLAUDE.md)
.root <- getwd()
while (!file.exists(file.path(.root, 'CLAUDE.md')) && dirname(.root) != .root)
  .root <- dirname(.root)
setwd(.root)
rm(.root)

suppressPackageStartupMessages({
  library(tidyverse)
  library(gt)
})

dir.create('results/tables', recursive = TRUE, showWarnings = FALSE)

PARAM_ORDER <- c(
  'Intercept',
  'H. Displacement',
  'V. Displacement',
  'Proj. plate_x',
  'Proj. plate_z',
  'Release Speed',
  'Balls',
  'Strikes',
  'Timing Offset'
)

RESP_ORDER <- c('VAA', 'HAA', 'Tilt')

TREATMENT_ROWS <- c('H. Displacement', 'V. Displacement')

# Mediator Models

In [5]:
med <- read_csv('results/tables/mediator_coefficients.csv', show_col_types = FALSE) |>
  mutate(
    param_label = factor(param_label, levels = PARAM_ORDER),
    resp_label  = factor(resp_label,  levels = RESP_ORDER)
  ) |>
  filter(!is.na(param_label)) |>
  arrange(param_label, resp_label)

med_wide <- med |>
  pivot_wider(
    id_cols     = param_label,
    names_from  = resp_label,
    values_from = c(coef, lower, upper),
    names_glue  = '{resp_label}_{.value}'
  )

resp_cols <- function(resp) {
  c(paste0(resp, '_coef'), paste0(resp, '_lower'), paste0(resp, '_upper'))
}

all_data_cols <- unlist(lapply(RESP_ORDER, resp_cols))

tbl_med <- med_wide |>
  select(param_label, all_of(all_data_cols)) |>
  gt(rowname_col = 'param_label') |>
  tab_stubhead(label = html('<b>Parameter</b>')) |>
  tab_spanner(label = 'VAA (Deg)',  columns = resp_cols('VAA'),  id = 'sp_vaa')  |>
  tab_spanner(label = 'HAA (Deg)',  columns = resp_cols('HAA'),  id = 'sp_haa')  |>
  tab_spanner(label = 'Tilt (Deg)', columns = resp_cols('Tilt'), id = 'sp_tilt') |>
  cols_label(
    VAA_coef   = html('<b>Coef</b>'),  VAA_lower  = html('<b>Lower</b>'),  VAA_upper  = html('<b>Upper</b>'),
    HAA_coef   = html('<b>Coef</b>'),  HAA_lower  = html('<b>Lower</b>'),  HAA_upper  = html('<b>Upper</b>'),
    Tilt_coef  = html('<b>Coef</b>'),  Tilt_lower = html('<b>Lower</b>'),  Tilt_upper = html('<b>Upper</b>')
  ) |>
  fmt_number(columns = everything(), decimals = 3) |>
  # left border between metric groups
  tab_style(
    style     = cell_borders(sides = 'left', color = '#aaaaaa', weight = px(1)),
    locations = list(
      cells_body(columns = c(resp_cols('VAA')[1], resp_cols('HAA')[1], resp_cols('Tilt')[1])),
      cells_column_labels(columns = c(resp_cols('VAA')[1], resp_cols('HAA')[1], resp_cols('Tilt')[1]))
    )
  ) |>
  # highlight treatment rows
  tab_style(
    style     = list(cell_fill(color = '#eef2ff'), cell_text(weight = 'bold')),
    locations = cells_stub(rows = param_label %in% TREATMENT_ROWS)
  ) |>
  tab_style(
    style     = cell_fill(color = '#eef2ff'),
    locations = cells_body(rows = param_label %in% TREATMENT_ROWS)
  ) |>
  # color positive/negative coef values
  tab_style(
    style     = cell_text(color = '#27ae60', weight = 'bold'),
    locations = cells_body(
      columns = c(VAA_coef, HAA_coef, Tilt_coef),
      rows    = VAA_coef > 0
    )
  ) |>
  tab_style(
    style     = cell_text(color = '#c0392b', weight = 'bold'),
    locations = cells_body(
      columns = c(VAA_coef, HAA_coef, Tilt_coef),
      rows    = VAA_coef < 0
    )
  ) |>
  tab_style(
    style     = cell_text(style = 'italic'),
    locations = cells_stub(rows = !(param_label %in% TREATMENT_ROWS))
  ) |>
  tab_style(
    style     = cell_text(weight = 'bold'),
    locations = cells_column_spanners()
  ) |>
  tab_source_note(
    source_note = 'Coef = mixed-effects OLS estimate; Lower/Upper = 95% CI. Blue rows = treatment (post-commit displacement).'
  ) |>
  tab_options(
    table.font.size                   = px(13),
    column_labels.font.size           = px(12),
    table.border.top.style            = 'solid',
    table.border.top.width            = px(2),
    table.border.top.color            = 'black',
    table_body.border.bottom.style    = 'solid',
    table_body.border.bottom.width    = px(2),
    table_body.border.bottom.color    = 'black',
    column_labels.border.top.style    = 'none',
    column_labels.border.bottom.style = 'solid',
    column_labels.border.bottom.width = px(1),
    column_labels.border.bottom.color = 'black',
    table.background.color            = 'white',
    stub.border.style                 = 'solid',
    stub.border.width                 = px(1),
    stub.border.color                 = '#aaaaaa'
  )

IRdisplay::display_html(as_raw_html(tbl_med))
tryCatch(
  gtsave(tbl_med, 'results/tables/mediator_coefficients.html'),
  error = function(e) message('gt HTML save skipped: ', conditionMessage(e))
)

Coef = mixed-effects OLS estimate; Lower/Upper = 95% CI. Blue rows = treatment (post-commit displacement).


ERROR: Error: Chromote: timed out waiting for response to command Target.createTarget


## Load Validation Data

In [ ]:
cm_metrics <- read_csv('data/viz_cache/cm_oos_metrics.csv',      show_col_types = FALSE)
cm_bip_cal <- read_csv('data/viz_cache/cm_bip_calibration.csv',  show_col_types = FALSE)
cm_foul_cal<- read_csv('data/viz_cache/cm_foul_calibration.csv', show_col_types = FALSE)
cm_ev      <- read_csv('data/viz_cache/cm_ev_scatter.csv',        show_col_types = FALSE)
cm_la      <- read_csv('data/viz_cache/cm_la_scatter.csv',        show_col_types = FALSE)
cm_cf      <- read_csv('data/viz_cache/cm_counterfactual.csv',    show_col_types = FALSE)

cat(sprintf(
  'Loaded | metrics: %d | bip_cal: %d | ev: %d | cf: %d\n',
  nrow(cm_metrics), nrow(cm_bip_cal), nrow(cm_ev), nrow(cm_cf)
))


## Outcome Model Out-of-Sample Performance (2025 Holdout)

In [ ]:
suppressPackageStartupMessages(library(scales))

# ── Classifier panel: AUC bars ──────────────────────────────────────────────
auc_df <- cm_metrics |>
  filter(metric == 'AUC-ROC') |>
  mutate(model = factor(model, levels = c('BIP classifier', 'Foul classifier')))

p_auc <- ggplot(auc_df, aes(x = model, y = model_val, fill = model)) +
  geom_col(width = 0.5) +
  geom_hline(yintercept = 0.5, linetype = 'dashed', color = 'grey40', linewidth = 0.8) +
  geom_text(aes(label = sprintf('AUC = %.3f', model_val)), vjust = -0.5,
            size = 5, fontface = 'bold') +
  scale_fill_manual(values = c('BIP classifier' = '#2980b9', 'Foul classifier' = '#27ae60'),
                    guide = 'none') +
  scale_y_continuous(limits = c(0, 1), labels = function(x) sprintf('%.2f', x),
                     expand = expansion(mult = c(0, 0.12))) +
  annotate('text', x = 2.4, y = 0.52, label = 'Random baseline', hjust = 1,
           size = 3.5, color = 'grey50') +
  labs(title = 'Classifier AUC-ROC (2025 holdout)', x = NULL, y = 'AUC-ROC') +
  theme(plot.title = element_text(size = 15))

# ── Regressor panel: R² bars ─────────────────────────────────────────────────
r2_df <- cm_metrics |>
  filter(metric == 'R²') |>
  mutate(model = factor(model, levels = c('Exit velo (XGB)', 'Launch angle (XGB)', 'EV/LA -> RV (XGB)')))

p_r2 <- ggplot(r2_df, aes(x = model, y = model_val, fill = model)) +
  geom_col(width = 0.5) +
  geom_hline(yintercept = 0, linetype = 'dashed', color = 'grey40', linewidth = 0.8) +
  geom_text(aes(label = sprintf('R² = %.3f', model_val)), vjust = -0.5,
            size = 5, fontface = 'bold') +
  scale_fill_manual(values = c(
    'Exit velo (XGB)'    = '#e74c3c',
    'Launch angle (XGB)' = '#e67e22',
    'EV/LA -> RV (XGB)'  = '#8e44ad'
  ), guide = 'none') +
  scale_y_continuous(limits = c(-0.05, 1), labels = function(x) sprintf('%.2f', x),
                     expand = expansion(mult = c(0.05, 0.12))) +
  labs(title = 'Regressor R² vs predict-mean baseline (2025 holdout)',
       x = NULL, y = 'R²') +
  theme(plot.title = element_text(size = 15),
        axis.text.x = element_text(size = 11))

options(repr.plot.width = 14, repr.plot.height = 6)
p_auc | p_r2
ggsave('results/plots/diagnostics/cm_oos_performance.png', p_auc | p_r2,
       width = 14, height = 6, dpi = 150)


## Classifier Calibration

In [ ]:
cal_df <- bind_rows(
  cm_bip_cal  |> mutate(model = 'BIP P(ball in play)'),
  cm_foul_cal |> mutate(model = 'Foul P(foul | not BIP)')
)

p_cal <- ggplot(cal_df, aes(x = mean_predicted, y = fraction_positive, color = model)) +
  geom_abline(slope = 1, intercept = 0, linetype = 'dashed', color = 'grey50', linewidth = 0.8) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3.5) +
  scale_color_manual(values = c('BIP P(ball in play)' = '#2980b9',
                                'Foul P(foul | not BIP)' = '#27ae60'),
                     name = NULL) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = 'Classifier Calibration: Predicted vs Actual Rates',
    subtitle = '2025 holdout  |  Dashed = perfect calibration  |  Quantile bins',
    x = 'Mean predicted probability',
    y = 'Observed fraction positive'
  ) +
  theme(legend.position = 'top')

options(repr.plot.width = 9, repr.plot.height = 7)
p_cal
ggsave('results/plots/diagnostics/cm_calibration.png', p_cal,
       width = 9, height = 7, dpi = 150)


## Exit Velocity & Launch Angle: Predicted vs Actual

In [ ]:
r2_ev <- 1 - sum((cm_ev$actual - cm_ev$predicted)^2) / sum((cm_ev$actual - mean(cm_ev$actual))^2)
r2_la <- 1 - sum((cm_la$actual - cm_la$predicted)^2) / sum((cm_la$actual - mean(cm_la$actual))^2)

p_ev <- ggplot(cm_ev, aes(x = predicted, y = actual)) +
  geom_point(alpha = 0.08, size = 0.7, color = '#e74c3c') +
  geom_abline(slope = 1, intercept = 0, color = 'black', linewidth = 0.9) +
  geom_smooth(method = 'lm', se = FALSE, color = '#2980b9', linewidth = 1.0, linetype = 'dashed') +
  annotate('text', x = -Inf, y = Inf, hjust = -0.1, vjust = 1.5,
           label = sprintf('R² = %.3f', r2_ev), size = 5, fontface = 'bold', color = '#c0392b') +
  labs(title = 'Exit Velocity: Predicted vs Actual',
       subtitle = 'XGBoost on BIP swings  |  2025 holdout  |  Black = perfect, blue = fit',
       x = 'Predicted EV (mph)', y = 'Actual EV (mph)') +
  theme(plot.title = element_text(size = 15))

p_la <- ggplot(cm_la, aes(x = predicted, y = actual)) +
  geom_point(alpha = 0.08, size = 0.7, color = '#e67e22') +
  geom_abline(slope = 1, intercept = 0, color = 'black', linewidth = 0.9) +
  geom_smooth(method = 'lm', se = FALSE, color = '#2980b9', linewidth = 1.0, linetype = 'dashed') +
  annotate('text', x = -Inf, y = Inf, hjust = -0.1, vjust = 1.5,
           label = sprintf('R² = %.3f', r2_la), size = 5, fontface = 'bold', color = '#c0392b') +
  labs(title = 'Launch Angle: Predicted vs Actual',
       subtitle = 'XGBoost on BIP swings  |  2025 holdout  |  Black = perfect, blue = fit',
       x = 'Predicted LA (deg)', y = 'Actual LA (deg)') +
  theme(plot.title = element_text(size = 15))

options(repr.plot.width = 14, repr.plot.height = 6)
p_ev | p_la
ggsave('results/plots/diagnostics/cm_ev_la_scatter.png', p_ev | p_la,
       width = 14, height = 6, dpi = 150)


## Counterfactual Sanity Check: Disruption Tax Distribution

In [ ]:
# Reconstruct implied xrv_intended from disruption_tax
# disruption_tax = xrv_realized - xrv_intended -> intended = realized - disruption_tax
# We don't have xrv_realized directly but disruption_tax is the gap

tax_long <- cm_cf |>
  select(disruption_tax, distortion_tax, selection_tax) |>
  pivot_longer(everything(), names_to = 'component', values_to = 'value') |>
  mutate(component = recode(component,
    'disruption_tax'  = 'Disruption tax (total)',
    'distortion_tax'  = 'Distortion tax',
    'selection_tax'   = 'Selection tax'
  ))

means <- tax_long |>
  group_by(component) |>
  summarise(m = mean(value), .groups = 'drop')

p_cf <- ggplot(tax_long, aes(x = value, fill = component, color = component)) +
  geom_density(alpha = 0.25, linewidth = 0.9) +
  geom_vline(data = means, aes(xintercept = m, color = component),
             linetype = 'dashed', linewidth = 0.9) +
  geom_vline(xintercept = 0, color = 'black', linewidth = 0.6) +
  scale_fill_manual(values  = c('Disruption tax (total)' = '#2c3e50',
                                'Distortion tax'         = '#e74c3c',
                                'Selection tax'          = '#2980b9'), name = NULL) +
  scale_color_manual(values = c('Disruption tax (total)' = '#2c3e50',
                                'Distortion tax'         = '#e74c3c',
                                'Selection tax'          = '#2980b9'), name = NULL) +
  scale_x_continuous(limits = c(-0.5, 0.3)) +
  labs(
    title    = 'Disruption Tax Components: Per-Swing Distribution (2025)',
    subtitle = paste0(
      'Disruption tax = distortion + selection  |  Dashed lines = component means  |  ',
      formatC(nrow(cm_cf), format = 'd', big.mark = ','), ' sampled swings'
    ),
    x = 'Tax (runs/swing, negative = pitcher advantage)',
    y = 'Density'
  ) +
  theme(legend.position = 'top')

options(repr.plot.width = 12, repr.plot.height = 6)
p_cf
ggsave('results/plots/diagnostics/cm_counterfactual_dist.png', p_cf,
       width = 12, height = 6, dpi = 150)
